In [ ]:
#r "nuget: FsLexYacc.Runtime, 11.3.0"
#r "Scanning/bin/Debug/net9.0/Scanning.dll"
open System
open System.IO
open LectureScanning
// Run python FSharp/tools/build.py from the repository root first.

# Evaluating an Exam Using FsLex

The scanner recognizes headers, maximum points, student names or matriculation numbers, and scores. A typed traversal computes grades; no state is attached dynamically to a lexer.

In [ ]:
let data = """Class: Algorithms and Complexity
          Group: TINF22AI1
          MaxPoints = 60
   
          Exercise:      1. 2. 3. 4. 5. 6.
          Jim Smith:     9 12 10  6  6  0
          John Slow:     4  4  2  0  -  -
          Susi Sorglos:  9 12 12  9  9  6
          1609922:       7  4 12  5  5  3
       """

This data show that there has been a exam with the subject <em style="color:blue">Algorithms and Complexity</em>
in the group <em style="color:blue">TIT22AI1</em>.  Furthermore, the equation
```
   MaxPoints = 60
```
shows that in order to achieve the best mark, <em style="color:blue">60</em> points would have been necessary.
    
There have been 6 different exercises in this exam and, in this small example,  only four students took part, namely *Jim Smith*, *John Slow*, *Susi Sorglos*, and some student that is only represented by their matriculation number.  Each of the rows decribing the results of the students begins with the name (or matriculation number) of the student followed by the number of points that they have achieved in the different exercises. Our goal is to write a program that is able to compute the marks for all students.

## Imports

The compiled FsLex scanner is referenced as a .NET assembly. Its actions return typed `ExamToken` values; F# transformations compute the grades separately from scanning.

## Auxiliary Functions

The function `mark(max_points, points)` takes two arguments:
- `points`     is the number of points achieved by the student whose mark is to be computed.
- `max_points` is the number of points that need to be achieved in order to get the best mark of $1.0$.
  
It is assumed that the relation between the mark of an exam and the number of points achieved in this exam is mostly linear and that a student who has achieved $50\%$ of `max_points` points will get the mark $4.0$, while a student who has achieved  $100\%$ of `max_points` points will get the mark $1.0$.  Therefore, the formula to calcuclate the grade is as follows:
$$ \textrm{grade} = 7 - 6 \cdot \frac{\texttt{points}}{\texttt{max_points}} $$
However, the worst mark is $5.0$.  Therefore, if the mark would fall below that line, the `min` function assures that it is less or equal than $5.0$. Furthermore, the resulting number is rounded to one digit.

In [ ]:
let mark maxPoints points =
    if maxPoints <= 0 then invalidArg "maxPoints" "Maximum must be positive"
    Math.Round(min 5.0 (7.0 - 6.0 * float points / float maxPoints), 1)

Plot the grading function as a portable SVG. The graph is written to `exam-grades.svg` and can be opened in a browser.

In [ ]:
let percentage maxPoints points =
    if maxPoints <= 0 then invalidArg "maxPoints" "Maximum must be positive"
    min 100.0 (Math.Round(100.0 * float points / float maxPoints))

In [ ]:
let plotGrade maxPoints =
    let coordinates = [0..maxPoints] |> List.map (fun p -> sprintf "%.2f,%.2f" (50.0 + 8.0*float p) (30.0 + 70.0*(mark maxPoints p - 1.0)))
    "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 600 360'><title>Exam grade as a function of points</title><path d='M50 20 V320 H550' fill='none' stroke='black'/><text x='250' y='350'>Points</text><text x='5' y='20'>Grade</text><text x='25' y='35'>1</text><text x='25' y='105'>2</text><text x='25' y='175'>3</text><text x='25' y='245'>4</text><text x='25' y='315'>5</text><text x='45' y='335'>0</text><text x='285' y='335'>30</text><text x='525' y='335'>60</text><polyline fill='none' stroke='blue' points='" + String.concat " " coordinates + "'/></svg>"
let gradePlot = plotGrade 60
File.WriteAllText("exam-grades.svg", gradePlot)
gradePlot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(gradePlot), "text/html") |> ignore

## Token Declarations

`ExamToken` is a discriminated union: `Header`, `Maximum`, `Student`, `Points`, `Newline`, and `ExamEnd`. Every action has a known return type, and the evaluation match must account for every token case.

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
File.ReadAllText("Scanning/Types.fs") |> printfn "%s"

In [ ]:
printfn "%s" data

## Token Definitions

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

### The `HEADER` Token

`HEADER` matches class, group and exercise heading lines. The scanner returns `Header text`; the evaluator discards that descriptive metadata rather than counting its numbers as student scores.

In [ ]:
let rule19 = """| ("Class:" | "Group:" | "Exercise:") [^ '\n' '\r']* { Header(text lexbuf) }"""
printfn "%s" rule19

### The Token `MAXDEF`

`MAXDEF` recognizes `MaxPoints = n`. The FsLex action converts the decimal digits after `=` to an integer and returns `Maximum n`. Grade computation rejects a non-positive maximum.

In [ ]:
let rule21 = """| "MaxPoints" blank* '=' blank* natural { Maximum(int ((text lexbuf).Split('=').[1].Trim())) }"""
printfn "%s" rule21

### The Token Name

The scanner returns `Student name` for letters, spaces and hyphens followed by `:`. The evaluation traversal finalizes a previous row if needed and starts a new point total.

In [ ]:
let rule23 = """| digit digit digit digit digit digit digit ':' { Student((text lexbuf).TrimEnd(':')) }
| letter [ 'A'-'Z' 'a'-'z' ' ' '-']* ':' { Student((text lexbuf).TrimEnd(':').Trim()) }"""
printfn "%s" rule23

### The Token `MATRICULATION`

Seven decimal digits followed by a colon are returned as a `Student` token, retaining the matriculation number as a string so leading zeroes are preserved.

In [ ]:
let rule25 = """| digit digit digit digit digit digit digit ':' { Student((text lexbuf).TrimEnd(':')) }"""
printfn "%s" rule25

### The Token `NUMBER`

Natural-number tokens return `Points n`. During evaluation, scores are added only while a student is active; header and exercise metadata cannot accidentally contribute points.

In [ ]:
let rule27 = """| "MaxPoints" blank* '=' blank* natural { Maximum(int ((text lexbuf).Split('=').[1].Trim())) }
| natural { Points(int (text lexbuf)) }"""
printfn "%s" rule27

### The Token `IGNORE`

Blank lines produce newline tokens but have no effect when no student is active. Spaces, tabs, carriage returns and standalone hyphens are skipped by the generated scanner.

In [ ]:
let rule29 = """| ("Class:" | "Group:" | "Exercise:") [^ '\n' '\r']* { Header(text lexbuf) }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; Newline }"""
printfn "%s" rule29

### The Token `LINEBREAK`

Each newline updates the source position and returns `Newline`. The evaluation traversal finishes the current student row, calculating points, grade and percentage. It also flushes the final row at end of input, even if the file has no final newline.

In [ ]:
type ExamResult = { Name: string; Points: int; Grade: float; Percentage: float }
let evaluate tokens =
    let mutable maximum = 0
    let mutable name = None
    let mutable points = 0
    let results = ResizeArray<ExamResult>()
    let finish () =
        match name with
        | Some student ->
            results.Add {Name=student; Points=points; Grade=mark maximum points; Percentage=percentage maximum points}
            name <- None
            points <- 0
        | None -> ()
    for token in tokens do
        match token with
        | Header _ -> ()
        | Maximum value -> maximum <- value
        | Student student -> finish (); name <- Some student
        | Points value when name.IsSome -> points <- points + value
        | Points _ -> ()
        | Newline | ExamEnd -> finish ()
    finish () // Also flush a final student line without a newline.
    List.ofSeq results

### Ignoring Characters

The FsLex character class below skips spaces, tabs, carriage returns and standalone hyphens. A hyphen inside a student name remains part of its longer student-name match.

In [ ]:
let rule33 = """let blank = [' ' '\t']
| [' ' '\t' '\r' '-']+ { token lexbuf }"""
printfn "%s" rule33

### Error Handling

An unmatched character raises an error with its input line. Malformed exam scores are reported rather than silently producing a misleading grade.

In [ ]:
let rule35 = """| _ { failwithf "Illegal exam character %s at line %d" (text lexbuf) lexbuf.StartPos.Line }"""
printfn "%s" rule35

### Build-time scanner generation

The adjacent project generates scanner code before notebook execution. FsLex does not inspect notebook filenames.

### Generating the Scanner and Running It

The next line generates the scanner.

In [ ]:
let scan = Scanning.exam

Next, we feed an input string into the generated scanner.

In [ ]:
let examTokens = scan data

In order to scan the data that we provided in the last line, the function `scan` iterates
over all tokens generated by our scanner.

In [ ]:
let results = evaluate examTokens

Finally, we can run the scanner.

In [ ]:
for row in results do printfn "%s: %d points, mark %.1f" row.Name row.Points row.Grade